# RetailMind AI — Day 13: Transformers & BERT
## Review Sentiment and Support-Ticket Routing

**Objective:** fine-tune a pretrained BERT model for Amazon review sentiment, create support-ticket labels from review text, train a second BERT classifier for ticket routing, and compare the transformer with a TF-IDF baseline.

> This notebook is designed for a 60-minute lab. It uses a reproducible subset of Amazon Polarity rather than downloading and training on the entire 3.6-million-row dataset. For a GPU, use the default sample sizes; on CPU, reduce them further.

## 1. Setup

The Hugging Face `datasets` package loads the Amazon Polarity dataset; `transformers` provides the pretrained BERT tokenizer and model. Scikit-learn is used only for the classical comparison and evaluation metrics.

In [ ]:
# Uncomment if these packages are missing in your environment:
# %pip install -U transformers datasets accelerate scikit-learn pandas numpy matplotlib torch

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## 2. Load Amazon Polarity

Amazon Polarity contains review text and binary labels: `0 = negative`, `1 = positive`. We take a stratified subset so the lab remains practical. If memory or runtime is limited, reduce `TRAIN_SAMPLES` and `TEST_SAMPLES`.

In [ ]:
TRAIN_SAMPLES = 12000
TEST_SAMPLES = 3000

# The dataset has predefined train/test splits.
raw = load_dataset("fancyzhx/amazon_polarity")

train_df = raw["train"].shuffle(seed=SEED).select(range(min(TRAIN_SAMPLES, len(raw["train"])))).to_pandas()
test_df = raw["test"].shuffle(seed=SEED).select(range(min(TEST_SAMPLES, len(raw["test"])))).to_pandas()

def combine_text(row):
    title = str(row.get("title", "") or "")
    content = str(row.get("content", "") or "")
    return (title + " " + content).strip()

train_df["text"] = train_df.apply(combine_text, axis=1)
test_df["text"] = test_df.apply(combine_text, axis=1)
train_df = train_df[["text", "label"]].dropna().reset_index(drop=True)
test_df = test_df[["text", "label"]].dropna().reset_index(drop=True)

print("Train:", train_df.shape, "Test:", test_df.shape)
print(train_df["label"].value_counts(normalize=True))
train_df.head()

## 3. Classical baseline — TF-IDF + Logistic Regression

This provides a consistent baseline for this notebook. If your Day-12 experiment report contains a TF-IDF score, record it in the comparison section as well. Use the same train/test subset for a fair comparison.

In [ ]:
tfidf_baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1))
])

tfidf_baseline.fit(train_df["text"], train_df["label"])
tfidf_pred = tfidf_baseline.predict(test_df["text"])
tfidf_accuracy = accuracy_score(test_df["label"], tfidf_pred)

print("TF-IDF accuracy:", round(tfidf_accuracy, 4))
print(classification_report(test_df["label"], tfidf_pred, target_names=["negative", "positive"]))

## 4. Tokenize text with BERT

We use `bert-base-uncased`. The tokenizer converts text into token IDs and attention masks. `MAX_LENGTH` limits sequence size to control memory and training time.

In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LENGTH):
        self.texts = frame["text"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            max_length=max_length,
            padding=False
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

review_train_ds = ReviewDataset(train_df, tokenizer)
review_test_ds = ReviewDataset(test_df, tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenizer("This product is excellent!", truncation=True, max_length=MAX_LENGTH))

## 5. Fine-tune BERT for review sentiment

The model has two output classes. Training is intentionally limited to one epoch for the lab. On CPU, this can still be slow: reduce the sample sizes or run on a GPU runtime if available.

In [ ]:
def make_training_args(output_dir, epochs=1, train_bs=8, eval_bs=16):
    # Evaluation/save argument names differ across Transformers releases.
    common = dict(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **common)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=2
)

sentiment_args = make_training_args("outputs/bert-sentiment", epochs=1)
sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_args,
    train_dataset=review_train_ds,
    eval_dataset=review_test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

sentiment_trainer.train()
sentiment_metrics = sentiment_trainer.evaluate()
print(sentiment_metrics)

## 6. Evaluate sentiment and compare with TF-IDF

In [ ]:
sentiment_output = sentiment_trainer.predict(review_test_ds)
bert_sentiment_pred = np.argmax(sentiment_output.predictions, axis=1)
bert_sentiment_accuracy = accuracy_score(test_df["label"], bert_sentiment_pred)

print("BERT accuracy:", round(bert_sentiment_accuracy, 4))
print(classification_report(
    test_df["label"], bert_sentiment_pred,
    target_names=["negative", "positive"]
))
print("Confusion matrix:\n", confusion_matrix(test_df["label"], bert_sentiment_pred))

comparison = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", "accuracy": tfidf_accuracy},
    {"model": "BERT fine-tuned", "accuracy": bert_sentiment_accuracy},
])
comparison

## 7. Create support-ticket labels from review text

The lab brief asks for ticket categories relabeled from reviews. Since Amazon Polarity does not contain real support-ticket labels, this notebook creates **weak labels** using transparent keyword rules. These are synthetic proxy labels, not ground-truth customer-service annotations.

Categories:
- `delivery`: shipping, arrival, packaging, tracking
- `billing`: price, payment, charge, refund
- `product_quality`: broken, defective, quality, durability
- `account_support`: login, account, password, customer service

Rows that do not match a category are excluded. Review the rules and examples before treating the results as meaningful.

In [ ]:
import re

TICKET_KEYWORDS = {
    "delivery": [
        "shipping", "shipped", "delivery", "delivered", "arrived",
        "late", "package", "packaging", "tracking", "shipment"
    ],
    "billing": [
        "price", "pricing", "charged", "charge", "payment", "refund",
        "money back", "billing", "expensive", "cost"
    ],
    "product_quality": [
        "broken", "defective", "poor quality", "quality", "durable",
        "broke", "damaged", "stopped working", "faulty", "replacement"
    ],
    "account_support": [
        "customer service", "support", "account", "login", "password",
        " sign in", "contacted", "representative", "help desk"
    ],
}

def assign_ticket_category(text):
    text = str(text).lower()
    scores = {
        category: sum(1 for phrase in phrases if phrase in text)
        for category, phrases in TICKET_KEYWORDS.items()
    }
    best_category = max(scores, key=scores.get)
    if scores[best_category] == 0:
        return None
    # Deterministic tie handling follows dictionary order.
    return best_category

ticket_df = pd.concat([train_df, test_df], ignore_index=True)
ticket_df["category"] = ticket_df["text"].apply(assign_ticket_category)
ticket_df = ticket_df.dropna(subset=["category"]).reset_index(drop=True)

print("Weakly labelled tickets:", len(ticket_df))
print(ticket_df["category"].value_counts())
ticket_df[["text", "category"]].head(10)

## 8. Train/test split and TF-IDF ticket-routing baseline

We use a stratified split where possible. Very small categories may not support a stratified split; inspect the category counts and collect more examples or simplify the category scheme if necessary.

In [ ]:
MIN_CATEGORY_ROWS = 20
valid_categories = ticket_df["category"].value_counts()
valid_categories = valid_categories[valid_categories >= MIN_CATEGORY_ROWS].index
ticket_df = ticket_df[ticket_df["category"].isin(valid_categories)].reset_index(drop=True)

ticket_labels = sorted(ticket_df["category"].unique())
label2id = {label: i for i, label in enumerate(ticket_labels)}
id2label = {i: label for label, i in label2id.items()}
ticket_df["label_id"] = ticket_df["category"].map(label2id)

ticket_train, ticket_test = train_test_split(
    ticket_df,
    test_size=0.2,
    random_state=SEED,
    stratify=ticket_df["label_id"]
)

ticket_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=40000, ngram_range=(1, 2), min_df=2, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
ticket_tfidf.fit(ticket_train["text"], ticket_train["category"])
ticket_tfidf_pred = ticket_tfidf.predict(ticket_test["text"])

print("Ticket TF-IDF accuracy:", round(accuracy_score(ticket_test["category"], ticket_tfidf_pred), 4))
print(classification_report(ticket_test["category"], ticket_tfidf_pred, zero_division=0))

## 9. Fine-tune the same BERT backbone for ticket routing

This is a multi-class classifier. The output layer is resized to the number of ticket categories. The labels are weak labels generated by keyword rules, so the score measures agreement with those rules—not validated real-world routing quality.

In [ ]:
ticket_train = ticket_train.reset_index(drop=True)
ticket_test = ticket_test.reset_index(drop=True)

class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LENGTH):
        self.labels = frame["label_id"].astype(int).tolist()
        self.encodings = tokenizer(
            frame["text"].astype(str).tolist(),
            truncation=True,
            max_length=max_length,
            padding=False
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

ticket_train_ds = TicketDataset(ticket_train, tokenizer)
ticket_test_ds = TicketDataset(ticket_test, tokenizer)

ticket_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(ticket_labels),
    id2label=id2label,
    label2id=label2id,
)

ticket_args = make_training_args("outputs/bert-ticket-routing", epochs=1)
ticket_trainer = Trainer(
    model=ticket_model,
    args=ticket_args,
    train_dataset=ticket_train_ds,
    eval_dataset=ticket_test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

ticket_trainer.train()
print(ticket_trainer.evaluate())

## 10. Evaluate ticket routing per class

In [ ]:
ticket_output = ticket_trainer.predict(ticket_test_ds)
ticket_bert_pred_ids = np.argmax(ticket_output.predictions, axis=1)
ticket_bert_pred = [id2label[int(i)] for i in ticket_bert_pred_ids]

ticket_bert_accuracy = accuracy_score(ticket_test["category"], ticket_bert_pred)
print("BERT ticket-routing accuracy:", round(ticket_bert_accuracy, 4))
print(classification_report(
    ticket_test["category"], ticket_bert_pred,
    labels=ticket_labels,
    zero_division=0
))
print("TF-IDF ticket-routing accuracy:",
      round(accuracy_score(ticket_test["category"], ticket_tfidf_pred), 4))

## 11. Save models and replace the classical sentiment module

The saved directories contain the model weights and tokenizer. In the production pipeline, call the BERT inference function instead of the Day-12 classical sentiment predictor. Keep the TF-IDF model only as a benchmark/fallback until deployment latency and memory are measured.

In [ ]:
ARTIFACT_DIR = "artifacts/day13"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

sentiment_dir = os.path.join(ARTIFACT_DIR, "bert_sentiment")
ticket_dir = os.path.join(ARTIFACT_DIR, "bert_ticket_router")

sentiment_trainer.save_model(sentiment_dir)
tokenizer.save_pretrained(sentiment_dir)
ticket_trainer.save_model(ticket_dir)
tokenizer.save_pretrained(ticket_dir)

def predict_sentiment(texts, model_dir=sentiment_dir):
    # Load once in a long-running service; do not reload for every request.
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    tok = AutoTokenizer.from_pretrained(model_dir)
    model.to(DEVICE)
    model.eval()
    batch = tok(list(texts), return_tensors="pt", truncation=True,
                padding=True, max_length=MAX_LENGTH)
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1)
    ids = probs.argmax(dim=-1).cpu().tolist()
    scores = probs.max(dim=-1).values.cpu().tolist()
    names = ["negative", "positive"]
    return [{"sentiment": names[i], "confidence": float(s)}
            for i, s in zip(ids, scores)]

def predict_ticket_category(texts, model_dir=ticket_dir):
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    tok = AutoTokenizer.from_pretrained(model_dir)
    model.to(DEVICE)
    model.eval()
    batch = tok(list(texts), return_tensors="pt", truncation=True,
                padding=True, max_length=MAX_LENGTH)
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**batch).logits, dim=-1)
    ids = probs.argmax(dim=-1).cpu().tolist()
    scores = probs.max(dim=-1).values.cpu().tolist()
    return [{"category": id2label[i], "confidence": float(s)}
            for i, s in zip(ids, scores)]

examples = [
    "The item arrived three days late and the package was damaged.",
    "The product is excellent and works exactly as described."
]
print("Sentiment:", predict_sentiment(examples))
print("Ticket routing:", predict_ticket_category(examples))

## 12. Final comparison and experiment notes

Accuracy alone is not sufficient for imbalanced or weakly labelled classes. Review the per-class precision/recall/F1 above. BERT may improve language understanding, but it costs more memory, training time, and inference latency than TF-IDF. Do not claim a lift unless your measured results support it.

In [ ]:
final_results = pd.DataFrame([
    {"task": "Review sentiment", "model": "TF-IDF + Logistic Regression", "accuracy": tfidf_accuracy},
    {"task": "Review sentiment", "model": "BERT fine-tuned", "accuracy": bert_sentiment_accuracy},
    {"task": "Ticket routing", "model": "TF-IDF + Logistic Regression", "accuracy": accuracy_score(ticket_test["category"], ticket_tfidf_pred)},
    {"task": "Ticket routing", "model": "BERT fine-tuned", "accuracy": ticket_bert_accuracy},
])
final_results.to_csv(os.path.join(ARTIFACT_DIR, "day13_model_comparison.csv"), index=False)
final_results

## 13. Integration checklist

- [ ] Compare sentiment results with the actual Day-12 baseline report (same split and metric).
- [ ] Inspect weak ticket labels and correct keyword rules; ideally replace them with manually reviewed labels.
- [ ] Measure inference latency and memory before deploying BERT.
- [ ] Update the RetailMind sentiment service/module to load `artifacts/day13/bert_sentiment`.
- [ ] Route support messages through `artifacts/day13/bert_ticket_router`.
- [ ] Keep model version, tokenizer, label mapping, and preprocessing together.
- [ ] Commit the notebook, README, and comparison CSV; avoid committing large model weights unless your repository policy permits it.

**Important limitation:** The ticket categories in this lab are heuristic labels created from review text. For a production-quality ticket router, collect real support tickets and human-verified routing labels.